Final Dataset(5/6B tokens)_Download- https://huggingface.co/datasets/almanach/Biomed-Enriched

In [1]:
!pip -q install -U datasets transformers huggingface_hub tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 9.5 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
#Project Config
import os
import json
import time
import hashlib
import gzip
import shutil
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/SLM_Biomed")

TRAIN_DIR = PROJECT_DIR / "train"
TEST_DIR = PROJECT_DIR / "test"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
META_DIR = PROJECT_DIR / "metadata"

for d in [
    PROJECT_DIR,
    TRAIN_DIR,
    TEST_DIR,
    CHECKPOINT_DIR,
    META_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)

Project: /content/drive/MyDrive/SLM_Biomed


In [ ]:
#Manage Hugging Face cache in Colab
HF_CACHE = "/content/hf_cache"

os.environ["HF_HOME"] = HF_CACHE
os.environ["HF_HUB_CACHE"] = f"{HF_CACHE}/hub"
os.environ["HF_DATASETS_CACHE"] = f"{HF_CACHE}/datasets"

os.makedirs(HF_CACHE, exist_ok=True)

print("HF cache:", HF_CACHE)

HF cache: /content/hf_cache


Configuration:

In [ ]:
#Training     5,000,000,000 tokens
#Development  2,000,000,000 tokens
#Test           150,000,000 tokens
#For larger Train set of 300M parmeters
# Simply change FINAL_MODEL_PARAMS = 300_000_000

DATASET_NAME = "almanach/Biomed-Enriched"
DATASET_SPLIT = "commercial"

TOKENIZER_NAME = "stanford-crfm/BioMedLM"

# Final model assumption
FINAL_MODEL_PARAMS = 250_000_000

# Chinchulla-style experimental rule we're using
TOKENS_PER_PARAMETER = 20

TARGET_TRAIN_TOKENS = (
    FINAL_MODEL_PARAMS * TOKENS_PER_PARAMETER
)

TARGET_TEST_TOKENS = 150_000_000

# Development corpus for 100M model
TARGET_DEV_TOKENS = 2_000_000_000

# Article-level test fraction
TEST_FRACTION = 0.03

# Each completed shard contains approximately this many tokens
#SHARD_TARGET_TOKENS = 500_000_000
SHARD_TARGET_TOKENS = 250_000_000

# Quality filtering
MIN_EDUCATIONAL_SCORE = 3.5
MIN_LANGUAGE_SCORE = 0.95
MIN_BIOMEDICAL_SCORE = 0.90

print("Training target :", f"{TARGET_TRAIN_TOKENS:,}")
print("Development     :", f"{TARGET_DEV_TOKENS:,}")
print("Test target     :", f"{TARGET_TEST_TOKENS:,}")
print("Shard size      :", f"{SHARD_TARGET_TOKENS:,}")

Training target : 5,000,000,000
Development     : 2,000,000,000
Test target     : 150,000,000
Shard size      : 250,000,000


In [ ]:
#Load BioMedLM tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_NAME
)

# We are using the tokenizer for counting.
# Individual training sequences will later be
# packed/truncated to the model context length.
tokenizer.model_max_length = int(1e9)

print("Tokenizer:", TOKENIZER_NAME)
print("Vocabulary:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/876 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/267 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/602k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/276k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Tokenizer: stanford-crfm/BioMedLM
Vocabulary: 28895


In [ ]:
#Load Streaming Dataset
from datasets import load_dataset

dataset = load_dataset(
    DATASET_NAME,
    split=DATASET_SPLIT,
    streaming=True
)

print(dataset)
print("Number of source shards:", dataset.num_shards)

README.md:   0%|          | 0.00/19.2k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

IterableDataset({
    features: ['id', 'article_id', 'path', 'text', 'language', 'section_title', 'domain', 'document_type', 'educational_score', 'domain_scores', 'document_type_scores', 'language_score', 'authors', 'article_url', 'license_url'],
    num_shards: 26
})
Number of source shards: 26


In [ ]:
#Quality filter
def get_biomedical_score(row):

    scores = row.get("domain_scores")

    if scores is None:
        return 0.0

    try:
        return float(scores[0])
    except:
        return 0.0


def quality_filter(row):

    text = row.get("text", "")

    if not text:
        return False

    if len(text.strip()) < 200:
        return False

    if row.get("language") != "en":
        return False

    try:
        language_score = float(
            row.get("language_score", 0)
        )
    except:
        language_score = 0

    if language_score < MIN_LANGUAGE_SCORE:
        return False

    if row.get("domain") != "biomedical":
        return False

    if get_biomedical_score(row) < MIN_BIOMEDICAL_SCORE:
        return False

    try:
        educational_score = float(
            row.get("educational_score", 0)
        )
    except:
        educational_score = 0

    if educational_score < MIN_EDUCATIONAL_SCORE:
        return False

    return True

In [ ]:
# Deterministic article-level split
#This ensures all paragraphs from an article remain in the same split.
def article_split(article_id):

    digest = hashlib.sha256(
        str(article_id).encode("utf-8")
    ).hexdigest()

    value = int(
        digest[:12],
        16
    ) / float(16**12 - 1)

    if value < TEST_FRACTION:
        return "test"

    return "train"

In [ ]:
#Token counting function
def count_tokens(text):

    encoded = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False
    )

    return len(encoded["input_ids"])

In [ ]:
#Checkpoint utilities
CHECKPOINT_FILE = (
    CHECKPOINT_DIR / "extraction_checkpoint.json"
)


def save_checkpoint(
    dataset_state,
    train_tokens,
    test_tokens,
    train_shards,
    test_shards,
    source_records
):

    checkpoint = {

        "timestamp": time.time(),

        "train_tokens": train_tokens,
        "test_tokens": test_tokens,

        "train_shards": train_shards,
        "test_shards": test_shards,

        "source_records": source_records,

        "dataset_state": dataset_state,

        "config": {
            "dataset": DATASET_NAME,
            "split": DATASET_SPLIT,
            "tokenizer": TOKENIZER_NAME,
            "test_fraction": TEST_FRACTION
        }
    }

    tmp_file = CHECKPOINT_DIR / "checkpoint.tmp"

    with open(
        tmp_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            checkpoint,
            f,
            indent=2
        )

        f.flush()
        os.fsync(f.fileno())

    os.replace(
        tmp_file,
        CHECKPOINT_FILE
    )


def load_checkpoint():

    if not CHECKPOINT_FILE.exists():
        return None

    with open(
        CHECKPOINT_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)

In [ ]:
#Save corpus statistics
def save_stats(stats):

    stats_file = META_DIR / "corpus_stats.json"

    tmp_file = META_DIR / "corpus_stats.tmp"

    with open(
        tmp_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            stats,
            f,
            indent=2
        )

        f.flush()
        os.fsync(f.fileno())

    os.replace(
        tmp_file,
        stats_file
    )

In [ ]:
#save Configuration
config = {

    "dataset": DATASET_NAME,

    "dataset_split": DATASET_SPLIT,

    "tokenizer": TOKENIZER_NAME,

    "final_model_parameters": FINAL_MODEL_PARAMS,

    "tokens_per_parameter": TOKENS_PER_PARAMETER,

    "target_train_tokens": TARGET_TRAIN_TOKENS,

    "target_development_tokens": TARGET_DEV_TOKENS,

    "target_test_tokens": TARGET_TEST_TOKENS,

    "test_fraction": TEST_FRACTION,

    "shard_target_tokens": SHARD_TARGET_TOKENS,

    "min_educational_score": MIN_EDUCATIONAL_SCORE,

    "min_language_score": MIN_LANGUAGE_SCORE,

    "min_biomedical_score": MIN_BIOMEDICAL_SCORE
}

with open(
    META_DIR / "config.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )

print(json.dumps(config, indent=2))

{
  "dataset": "almanach/Biomed-Enriched",
  "dataset_split": "commercial",
  "tokenizer": "stanford-crfm/BioMedLM",
  "final_model_parameters": 250000000,
  "tokens_per_parameter": 20,
  "target_train_tokens": 5000000000,
  "target_development_tokens": 2000000000,
  "target_test_tokens": 150000000,
  "test_fraction": 0.03,
  "shard_target_tokens": 250000000,
  "min_educational_score": 3.5,
  "min_language_score": 0.95,
  "min_biomedical_score": 0.9
}


In [ ]:
import psutil

def get_ram_gb():

    process = psutil.Process(
        os.getpid()
    )

    return process.memory_info().rss / (1024 ** 3)


print(
    f"Current Python RAM: "
    f"{get_ram_gb():.2f} GB"
)

Current Python RAM: 0.56 GB


In [ ]:
#Production extractor
# ============================================================
# MEMORY-SAFE PRODUCTION EXTRACTOR
# ============================================================

import os
import gc
import json
import gzip
import time
import hashlib

from pathlib import Path
from tqdm.auto import tqdm
from datasets import load_dataset


# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

SHARD_TARGET_TOKENS = 250_000_000

CHECKPOINT_INTERVAL_SECONDS = 300

CHECKPOINT_FILE = (
    CHECKPOINT_DIR / "extraction_checkpoint.json"
)


# ------------------------------------------------------------
# ARTICLE-LEVEL SPLIT
# ------------------------------------------------------------

def article_split(article_id):

    digest = hashlib.sha256(
        str(article_id).encode("utf-8")
    ).hexdigest()

    value = int(digest[:12], 16) / float(16**12 - 1)

    if value < TEST_FRACTION:
        return "test"

    return "train"


# ------------------------------------------------------------
# TOKEN COUNT
# ------------------------------------------------------------

def count_tokens(text):

    encoded = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False
    )

    return len(encoded["input_ids"])


# ------------------------------------------------------------
# QUALITY FILTER
# ------------------------------------------------------------

def get_biomedical_score(row):

    scores = row.get("domain_scores")

    if scores is None:
        return 0.0

    try:
        return float(scores[0])
    except Exception:
        return 0.0


def quality_filter(row):

    text = row.get("text", "")

    if not text:
        return False

    if len(text.strip()) < 200:
        return False

    # English only
    if row.get("language") != "en":
        return False

    try:
        language_score = float(
            row.get("language_score", 0)
        )
    except Exception:
        language_score = 0

    if language_score < MIN_LANGUAGE_SCORE:
        return False

    if row.get("domain") != "biomedical":
        return False

    if get_biomedical_score(row) < MIN_BIOMEDICAL_SCORE:
        return False

    try:
        educational_score = float(
            row.get("educational_score", 0)
        )
    except Exception:
        educational_score = 0

    if educational_score < MIN_EDUCATIONAL_SCORE:
        return False

    return True


# ------------------------------------------------------------
# CHECKPOINT SAVE
# ------------------------------------------------------------

def save_checkpoint(
    dataset_state,
    train_tokens,
    test_tokens,
    train_shards,
    test_shards,
    source_records
):

    checkpoint = {

        "timestamp": time.time(),

        "train_tokens": train_tokens,

        "test_tokens": test_tokens,

        "train_shards": train_shards,

        "test_shards": test_shards,

        "source_records": source_records,

        "dataset_state": dataset_state,

        "config": {

            "dataset": DATASET_NAME,

            "split": DATASET_SPLIT,

            "tokenizer": TOKENIZER_NAME,

            "test_fraction": TEST_FRACTION,

            "shard_target_tokens":
                SHARD_TARGET_TOKENS
        }
    }

    tmp_file = (
        CHECKPOINT_DIR /
        "checkpoint.tmp"
    )

    with open(
        tmp_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            checkpoint,
            f,
            indent=2
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        tmp_file,
        CHECKPOINT_FILE
    )


# ------------------------------------------------------------
# CHECKPOINT LOAD
# ------------------------------------------------------------

def load_checkpoint():

    if not CHECKPOINT_FILE.exists():

        return None

    with open(
        CHECKPOINT_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


# ------------------------------------------------------------
# CREATE SHARD
# ------------------------------------------------------------

def open_shard(split, shard_number):

    if split == "train":

        directory = TRAIN_DIR

    else:

        directory = TEST_DIR

    directory.mkdir(
        parents=True,
        exist_ok=True
    )

    final_path = (
        directory /
        f"shard_{shard_number:05d}.jsonl.gz"
    )

    temporary_path = (
        directory /
        f".shard_{shard_number:05d}.tmp"
    )

    # If a previous interrupted run left a temp file,
    # remove it before starting the shard again.
    if temporary_path.exists():

        temporary_path.unlink()

    f = gzip.open(
        temporary_path,
        "wt",
        encoding="utf-8",
        compresslevel=6
    )

    return f, temporary_path, final_path


# ------------------------------------------------------------
# CLOSE / COMMIT SHARD
# ------------------------------------------------------------

def close_shard(
    file_handle,
    temporary_path,
    final_path
):

    file_handle.flush()

    file_handle.close()

    # Atomic rename.
    os.replace(
        temporary_path,
        final_path
    )

    # Explicit cleanup
    del file_handle

    gc.collect()


# ============================================================
# MAIN EXTRACTION
# ============================================================

def extract_production():

    checkpoint = load_checkpoint()

    # --------------------------------------------------------
    # RESUME OR START
    # --------------------------------------------------------

    if checkpoint is not None:

        print("=" * 70)
        print("CHECKPOINT FOUND")
        print("=" * 70)

        train_tokens = checkpoint["train_tokens"]

        test_tokens = checkpoint["test_tokens"]

        train_shards = checkpoint["train_shards"]

        test_shards = checkpoint["test_shards"]

        source_records = checkpoint["source_records"]

        print(
            "Train tokens:",
            f"{train_tokens:,}"
        )

        print(
            "Test tokens:",
            f"{test_tokens:,}"
        )

        print(
            "Completed train shards:",
            train_shards
        )

        print(
            "Completed test shards:",
            test_shards
        )

        dataset = load_dataset(
            DATASET_NAME,
            split=DATASET_SPLIT,
            streaming=True
        )

        # Restore source stream position
        dataset.load_state_dict(
            checkpoint["dataset_state"]
        )

        print(
            "✓ Streaming dataset position restored"
        )

    else:

        print("=" * 70)
        print("NO CHECKPOINT FOUND")
        print("STARTING NEW EXTRACTION")
        print("=" * 70)

        train_tokens = 0

        test_tokens = 0

        train_shards = 0

        test_shards = 0

        source_records = 0

        dataset = load_dataset(
            DATASET_NAME,
            split=DATASET_SPLIT,
            streaming=True
        )

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # We do NOT keep records in a giant Python list.
    #
    # Records go directly into gzip.
    # --------------------------------------------------------

    train_file = None
    train_tmp = None
    train_final = None

    test_file = None
    test_tmp = None
    test_final = None

    train_shard_tokens = 0
    test_shard_tokens = 0

    last_checkpoint = time.time()

    # --------------------------------------------------------
    # Existing shard protection
    # --------------------------------------------------------

    while (
        TRAIN_DIR /
        f"shard_{train_shards:05d}.jsonl.gz"
    ).exists():

        train_shards += 1

    while (
        TEST_DIR /
        f"shard_{test_shards:05d}.jsonl.gz"
    ).exists():

        test_shards += 1

    print(
        "Next train shard:",
        train_shards
    )

    print(
        "Next test shard:",
        test_shards
    )

    # --------------------------------------------------------
    # STREAM DATA
    # --------------------------------------------------------

    progress = tqdm(
        dataset,
        desc="Building corpus",
        unit="records"
    )

    for row in progress:

        source_records += 1

        # ----------------------------------------------------
        # Stop once targets are reached
        # ----------------------------------------------------

        if (
            train_tokens >= TARGET_TRAIN_TOKENS
            and
            test_tokens >= TARGET_TEST_TOKENS
        ):

            break

        # ----------------------------------------------------
        # QUALITY FILTER
        # ----------------------------------------------------

        if not quality_filter(row):

            continue

        article_id = row.get(
            "article_id"
        )

        if not article_id:

            continue

        # ----------------------------------------------------
        # ARTICLE-LEVEL SPLIT
        # ----------------------------------------------------

        split = article_split(
            article_id
        )

        # ----------------------------------------------------
        # TOKEN COUNT
        # ----------------------------------------------------

        token_count = count_tokens(
            row["text"]
        )

        if token_count == 0:

            continue

        # ----------------------------------------------------
        # TRAIN
        # ----------------------------------------------------

        if (
            split == "train"
            and
            train_tokens < TARGET_TRAIN_TOKENS
        ):

            # Open new shard when necessary
            if train_file is None:

                (
                    train_file,
                    train_tmp,
                    train_final
                ) = open_shard(
                    "train",
                    train_shards
                )

                train_shard_tokens = 0

                print(
                    f"\nOpened train shard "
                    f"{train_shards:05d}"
                )

            record = {

                "article_id": article_id,

                "text": row["text"],

                "token_count": token_count,

                "section_title":
                    row.get("section_title"),

                "document_type":
                    row.get("document_type"),

                "educational_score":
                    row.get("educational_score"),

                "article_url":
                    row.get("article_url"),

                "license_url":
                    row.get("license_url")
            }

            # DIRECT WRITE — no giant buffer
            train_file.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )

            train_tokens += token_count

            train_shard_tokens += token_count

            # Release record immediately
            del record

            # ------------------------------------------------
            # TRAIN SHARD COMPLETE
            # ------------------------------------------------

            if (
                train_shard_tokens
                >= SHARD_TARGET_TOKENS
            ):

                close_shard(
                    train_file,
                    train_tmp,
                    train_final
                )

                print(
                    f"\nCompleted train shard: "
                    f"{train_final.name}"
                )

                train_shards += 1

                train_file = None

                train_tmp = None

                train_final = None

                train_shard_tokens = 0

                # Aggressive memory cleanup
                gc.collect()

                # --------------------------------------------
                # CHECKPOINT
                # --------------------------------------------

                save_checkpoint(
                    dataset.state_dict(),

                    train_tokens,

                    test_tokens,

                    train_shards,

                    test_shards,

                    source_records
                )

                print(
                    f"CHECKPOINT: "
                    f"{train_tokens / 1e9:.3f}B "
                    f"train tokens"
                )

        # ----------------------------------------------------
        # TEST
        # ----------------------------------------------------

        elif (
            split == "test"
            and
            test_tokens < TARGET_TEST_TOKENS
        ):

            if test_file is None:

                (
                    test_file,
                    test_tmp,
                    test_final
                ) = open_shard(
                    "test",
                    test_shards
                )

                test_shard_tokens = 0

                print(
                    f"\nOpened test shard "
                    f"{test_shards:05d}"
                )

            record = {

                "article_id": article_id,

                "text": row["text"],

                "token_count": token_count,

                "section_title":
                    row.get("section_title"),

                "document_type":
                    row.get("document_type"),

                "educational_score":
                    row.get("educational_score"),

                "article_url":
                    row.get("article_url"),

                "license_url":
                    row.get("license_url")
            }

            test_file.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )

            test_tokens += token_count

            test_shard_tokens += token_count

            del record

            # ------------------------------------------------
            # TEST SHARD COMPLETE
            # ------------------------------------------------

            if (
                test_shard_tokens
                >= SHARD_TARGET_TOKENS
            ):

                close_shard(
                    test_file,
                    test_tmp,
                    test_final
                )

                print(
                    f"\nCompleted test shard: "
                    f"{test_final.name}"
                )

                test_shards += 1

                test_file = None

                test_tmp = None

                test_final = None

                test_shard_tokens = 0

                gc.collect()

                save_checkpoint(
                    dataset.state_dict(),

                    train_tokens,

                    test_tokens,

                    train_shards,

                    test_shards,

                    source_records
                )

                print(
                    f"CHECKPOINT: "
                    f"{test_tokens / 1e6:.1f}M "
                    f"test tokens"
                )

        # ----------------------------------------------------
        # PERIODIC CHECKPOINT
        # ----------------------------------------------------

        if (
            time.time() - last_checkpoint
            >= CHECKPOINT_INTERVAL_SECONDS
        ):

            # Flush open files so the data written so far
            # is physically pushed through the Python buffer.
            if train_file is not None:

                train_file.flush()

            if test_file is not None:

                test_file.flush()

            save_checkpoint(

                dataset.state_dict(),

                train_tokens,

                test_tokens,

                train_shards,

                test_shards,

                source_records
            )

            last_checkpoint = time.time()

        # ----------------------------------------------------
        # PROGRESS DISPLAY
        # ----------------------------------------------------

        progress.set_postfix({

            "train_B":
                f"{train_tokens / 1e9:.3f}",

            "test_M":
                f"{test_tokens / 1e6:.1f}",

            "train_shards":
                train_shards,

            "RAM_GB":
                f"{get_ram_gb():.1f}"
                if "get_ram_gb" in globals()
                else "?"
        })

    # ========================================================
    # FINALIZE OPEN SHARDS
    # ========================================================

    if train_file is not None:

        close_shard(
            train_file,
            train_tmp,
            train_final
        )

        print(
            f"Completed final train shard: "
            f"{train_final.name}"
        )

        train_shards += 1

    if test_file is not None:

        close_shard(
            test_file,
            test_tmp,
            test_final
        )

        print(
            f"Completed final test shard: "
            f"{test_final.name}"
        )

        test_shards += 1

    # --------------------------------------------------------
    # FINAL CLEANUP
    # --------------------------------------------------------

    train_file = None
    test_file = None

    train_tmp = None
    test_tmp = None

    train_final = None
    test_final = None

    gc.collect()

    # --------------------------------------------------------
    # FINAL STATISTICS
    # --------------------------------------------------------

    stats = {

        "dataset": DATASET_NAME,

        "tokenizer": TOKENIZER_NAME,

        "train_tokens": train_tokens,

        "test_tokens": test_tokens,

        "total_tokens":
            train_tokens + test_tokens,

        "train_shards":
            train_shards,

        "test_shards":
            test_shards,

        "source_records_processed":
            source_records,

        "target_train_tokens":
            TARGET_TRAIN_TOKENS,

        "target_test_tokens":
            TARGET_TEST_TOKENS,

        "target_development_tokens":
            TARGET_DEV_TOKENS,

        "shard_target_tokens":
            SHARD_TARGET_TOKENS,

        "train_target_reached":
            train_tokens >= TARGET_TRAIN_TOKENS,

        "test_target_reached":
            test_tokens >= TARGET_TEST_TOKENS
    }

    save_stats(stats)

    # --------------------------------------------------------
    # Remove checkpoint ONLY after successful completion
    # --------------------------------------------------------

    if (
        train_tokens >= TARGET_TRAIN_TOKENS
        and
        test_tokens >= TARGET_TEST_TOKENS
    ):

        if CHECKPOINT_FILE.exists():

            CHECKPOINT_FILE.unlink()

        print(
            "\nExtraction completed successfully."
        )

    else:

        # Keep checkpoint!
        print(
            "\nExtraction stopped before targets."
        )

        print(
            "Checkpoint retained for resume."
        )

    print("\n")
    print("=" * 70)
    print("CORPUS STATUS")
    print("=" * 70)

    print(
        f"Training tokens : "
        f"{train_tokens / 1e9:.3f}B"
    )

    print(
        f"Test tokens     : "
        f"{test_tokens / 1e6:.1f}M"
    )

    print(
        f"Train shards    : "
        f"{train_shards}"
    )

    print(
        f"Test shards     : "
        f"{test_shards}"
    )

    return stats

In [ ]:
#Run production Extractor
stats = extract_production()

CHECKPOINT FOUND
Train tokens: 850,288,070
Test tokens: 25,913,806
Completed train shards: 1
Completed test shards: 0


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

✓ Streaming dataset position restored
Next train shard: 1
Next test shard: 0


Building corpus: 0records [00:00, ?records/s]


Opened train shard 00001

Opened test shard 00000

Completed train shard: shard_00001.jsonl.gz
CHECKPOINT: 1.100B train tokens

Opened train shard 00002

Completed train shard: shard_00002.jsonl.gz
CHECKPOINT: 1.350B train tokens

Opened train shard 00003

Completed train shard: shard_00003.jsonl.gz
CHECKPOINT: 1.600B train tokens

Opened train shard 00004

Completed train shard: shard_00004.jsonl.gz
CHECKPOINT: 1.850B train tokens

Opened train shard 00005

Completed train shard: shard_00005.jsonl.gz
CHECKPOINT: 2.100B train tokens

Opened train shard 00006

Completed train shard: shard_00006.jsonl.gz
CHECKPOINT: 2.350B train tokens

Opened train shard 00007

Completed train shard: shard_00007.jsonl.gz
CHECKPOINT: 2.600B train tokens

Opened train shard 00008

Completed train shard: shard_00008.jsonl.gz
CHECKPOINT: 2.850B train tokens

Opened train shard 00009

Completed train shard: shard_00009.jsonl.gz
CHECKPOINT: 3.100B train tokens

Opened train shard 00010

Completed train shard

In [ ]:
#Check corpus after extraction
import os


def directory_size_gb(directory):

    total = 0

    for root, dirs, files in os.walk(directory):

        for file in files:

            path = os.path.join(
                root,
                file
            )

            try:
                total += os.path.getsize(path)
            except OSError:
                pass

    return total / (1024 ** 3)


print(
    "Train storage:",
    f"{directory_size_gb(TRAIN_DIR):.2f} GB"
)

print(
    "Test storage:",
    f"{directory_size_gb(TEST_DIR):.2f} GB"
)

print(
    "Total project:",
    f"{directory_size_gb(PROJECT_DIR):.2f} GB"
)

Train storage: 6.69 GB
Test storage: 0.19 GB
Total project: 6.88 GB


In [ ]:
#Verify Shards
train_shards = sorted(
    TRAIN_DIR.glob("shard_*.jsonl.gz")
)

test_shards = sorted(
    TEST_DIR.glob("shard_*.jsonl.gz")
)

print(
    "Train shards:",
    len(train_shards)
)

print(
    "Test shards:",
    len(test_shards)
)

print("\nFirst train shards:")

for f in train_shards[:5]:
    print(
        f.name,
        f"{f.stat().st_size / 1024**2:.1f} MB"
    )

Train shards: 18
Test shards: 1

First train shards:
shard_00000.jsonl.gz 745.3 MB
shard_00001.jsonl.gz 372.1 MB
shard_00002.jsonl.gz 369.5 MB
shard_00003.jsonl.gz 362.8 MB
shard_00004.jsonl.gz 363.2 MB


In [ ]:
#Verify if Shard is readable
import gzip
import json

sample_shard = train_shards[0]

with gzip.open(
    sample_shard,
    "rt",
    encoding="utf-8"
) as f:

    for i in range(3):

        record = json.loads(
            f.readline()
        )

        print(
            "Article:",
            record["article_id"]
        )

        print(
            "Tokens:",
            record["token_count"]
        )

        print(
            "Text:",
            record["text"][:200],
            "\n"
        )

Article: PMC11276146
Tokens: 171
Text: Differentiated microbiotic habitats originate in the oral cavity at the level of the hard tissues of the teeth and in the soft tissues of the gingival sulcus, the back of the tongue, the lips, the che 

Article: PMC11276146
Tokens: 668
Text: Dental caries is an infectious, degenerative and destructive disease affecting the hard substances of the tooth (the enamel and dentine); if not treated, they progress in depth up to the dental pulp.  

Article: PMC11276146
Tokens: 240
Text: Dental caries can have a negative impact on the health and quality of the young patient due to the pain, which causes a reduction in the appetite with consequent weight loss due to difficulties in che 



In [ ]:
#Create the 2B development manifest
DEV_TARGET = 2_000_000_000

dev_tokens = 0
dev_shards = []

for shard in train_shards:

    with gzip.open(
        shard,
        "rt",
        encoding="utf-8"
    ) as f:

        shard_tokens = 0

        for line in f:

            record = json.loads(line)

            shard_tokens += record["token_count"]

    if dev_tokens < DEV_TARGET:

        dev_shards.append(
            {
                "file": shard.name,
                "tokens": shard_tokens
            }
        )

        dev_tokens += shard_tokens

    else:

        break


dev_manifest = {

    "target_tokens": DEV_TARGET,

    "actual_tokens": dev_tokens,

    "shards": dev_shards
}

with open(
    META_DIR / "development_2B_manifest.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        dev_manifest,
        f,
        indent=2
    )

print(
    json.dumps(
        dev_manifest,
        indent=2
    )
)

{
  "target_tokens": 2000000000,
  "actual_tokens": 2000000956,
  "shards": [
    {
      "file": "shard_00000.jsonl.gz",
      "tokens": 500000080
    },
    {
      "file": "shard_00001.jsonl.gz",
      "tokens": 250000277
    },
    {
      "file": "shard_00002.jsonl.gz",
      "tokens": 250000044
    },
    {
      "file": "shard_00003.jsonl.gz",
      "tokens": 250000137
    },
    {
      "file": "shard_00004.jsonl.gz",
      "tokens": 250000077
    },
    {
      "file": "shard_00005.jsonl.gz",
      "tokens": 250000226
    },
    {
      "file": "shard_00006.jsonl.gz",
      "tokens": 250000115
    }
  ]
}


### Inspecting Training Data Shards

In [5]:
import gzip
import json
from pathlib import Path

def inspect_shards(shard_list, num_records=2, dataset_name="dataset"):
    print(f"\n--- Inspecting first {num_records} records from {len(shard_list)} {dataset_name} shards ---")

    for i, shard_path in enumerate(shard_list):
        print(f"\nShard: {shard_path.name}")
        try:
            with gzip.open(shard_path, "rt", encoding="utf-8") as f:
                records_read = 0
                for line in f:
                    if records_read >= num_records:
                        break

                    record = json.loads(line)
                    if records_read == 0: # Print headers only once per shard
                        print("Headers:", list(record.keys()))
                    print(f"Record {records_read + 1}:")
                    # Print some key fields, or the whole record if small enough
                    for k, v in record.items():
                        if isinstance(v, str) and len(v) > 100:
                            print(f"  {k}: {v[:100]}...")
                        else:
                            print(f"  {k}: {v}")
                    records_read += 1
        except Exception as e:
            print(f"Error reading shard {shard_path.name}: {e}")

# Define train_shards and test_shards here to ensure they are available
# This assumes TRAIN_DIR and TEST_DIR (from cell x68qS1qr7SQM) are in scope.
train_shards = sorted(
    TRAIN_DIR.glob("shard_*.jsonl.gz")
)

test_shards = sorted(
    TEST_DIR.glob("shard_*.jsonl.gz")
)

# Inspect training shards
inspect_shards(train_shards, num_records=2, dataset_name="train")


--- Inspecting first 2 records from 18 train shards ---

Shard: shard_00000.jsonl.gz
Headers: ['article_id', 'text', 'token_count', 'section_title', 'document_type', 'educational_score', 'article_url', 'license_url']
Record 1:
  article_id: PMC11276146
  text: Differentiated microbiotic habitats originate in the oral cavity at the level of the hard tissues of...
  token_count: 171
  section_title: 1. Introduction
  document_type: study
  educational_score: 4.08203125
  article_url: https://doi.org/10.3390/children11070816
  license_url: https://creativecommons.org/licenses/by/4.0/
Record 2:
  article_id: PMC11276146
  text: Dental caries is an infectious, degenerative and destructive disease affecting the hard substances o...
  token_count: 668
  section_title: 1.1. Etiology of Caries
  document_type: review
  educational_score: 4.96484375
  article_url: https://doi.org/10.3390/children11070816
  license_url: https://creativecommons.org/licenses/by/4.0/

Shard: shard_00001.jsonl.gz
He

### Inspecting Test Data Shards

In [6]:
# Inspect test shards
inspect_shards(test_shards, num_records=2, dataset_name="test")


--- Inspecting first 2 records from 1 test shards ---

Shard: shard_00000.jsonl.gz
Headers: ['article_id', 'text', 'token_count', 'section_title', 'document_type', 'educational_score', 'article_url', 'license_url']
Record 1:
  article_id: PMC11857926
  text: Emodin (EMO) is a naturally occurring compound that can be extracted from the roots and barks of var...
  token_count: 161
  section_title: 1. Introduction
  document_type: study
  educational_score: 4.125
  article_url: https://doi.org/10.3390/molecules30040822
  license_url: https://creativecommons.org/licenses/by/4.0/
Record 2:
  article_id: PMC11857926
  text: Drug–carrier compatibility is used to describe the miscibility of a drug–carrier system . Theoretica...
  token_count: 163
  section_title: 1. Introduction
  document_type: study
  educational_score: 4.1015625
  article_url: https://doi.org/10.3390/molecules30040822
  license_url: https://creativecommons.org/licenses/by/4.0/
